## Transform customer Data
### 1. Remove records with null customer_id
### 2. remove exact duplicate records
### 3. remove duplicate records based on creatd_timestamp
### 4. Cast the columns to the correct data type
### 5. Write transformed data to the silver schema

In [0]:

df_customer = spark.read.table("gizmobox_catalog_noori.bronze.py1_customers")
# display(df_customer)
df_distinct_customer = df_customer.filter(df_customer.customer_id.isNotNull()).distinct().orderBy("customer_id")
display(df_distinct_customer)


In [0]:
from pyspark.sql.functions import max
df_aggregate_customer = df_distinct_customer.groupBy("customer_id").agg(max("created_timestamp").alias("created_timestamp"),max("customer_name").alias("customer_name"),max("date_of_birth").alias("date_of_birth"),max("email").alias("email"),max("member_since").alias("member_since"),max("telephone").alias("telephone"))

display(df_aggregate_customer)

In [0]:
%sql
create or replace temporary view v_customers_distinct as
select distinct * 
from gizmobox_catalog_noori.bronze.v_customers
where customer_id is not null
order by customer_id

In [0]:
from pyspark.sql.functions import max
df_aggregate_customer = df_distinct_customer.groupBy("customer_id").agg(max("created_timestamp").alias("max_created_timestamp"))
# display(df_aggregate_customer)

df_aggregate_customer_join = (df_distinct_customer
    .join(df_aggregate_customer, (df_aggregate_customer.customer_id == df_distinct_customer.customer_id) & (df_aggregate_customer.max_created_timestamp == df_distinct_customer.created_timestamp),"inner").select(df_distinct_customer['*'])
)


display(df_aggregate_customer_join)

In [0]:
from pyspark.sql.functions import max
df_aggregate_customer_join_cast = df_aggregate_customer_join.select(df_aggregate_customer_join.created_timestamp.cast('timestamp'),df_aggregate_customer_join.customer_id,df_aggregate_customer_join.customer_name,df_aggregate_customer_join.date_of_birth.cast('date'),df_aggregate_customer_join.email,df_aggregate_customer_join.member_since.cast('date'),df_aggregate_customer_join.telephone)


display(df_aggregate_customer_join_cast)

In [0]:
df_aggregate_customer_join_cast.writeTo("gizmobox_catalog_noori.silver.py1_customers").createOrReplace()

In [0]:
spark.read.table("gizmobox_catalog_noori.silver.py1_customers").display()